# 第2回：Blueskyから投稿を集める（データ取得）

**このNotebookでできるようになること**

1. SNS「Bluesky」の公開APIから，キーワードや期間を指定して投稿（本文・投稿日時・反応数・画像）を集める
2. 集めたデータを表（DataFrame）として眺め，CSVに保存する
3. 特定のアカウントの投稿や，投稿への返信を集める
4. 投稿に付いた画像をダウンロードする

> **使い方**：上から順にセルを実行（`Shift + Enter`）して，結果を確認してください．
> 検索語（`"防災"` など）は自由に書き換えて試してかまいません．

## Blueskyとは

- X（旧Twitter）に似た短文SNS．投稿・いいね・リポスト・返信・画像投稿ができる．
- **公開されている投稿は「API」からプログラムで取得できる**．有料の API キー申請は不要で，無料のアカウントがあればよい．
- 投稿には本文，投稿日時，反応数（いいね・リポスト・返信・引用），画像，言語タグなどが付いている．

## APIとは

「プログラムからデータを取り出すための窓口」です．例えば次のURLは「`防災` を含む投稿を5件返して」という意味で，JSON形式の結果が返ってきます．

`https://api.bsky.app/xrpc/app.bsky.feed.searchPosts?q=防災&lang=ja&limit=5`

このNotebookでは，URLを組み立てる部分を `bsky_utils.py` の関数にまとめてあるので，**関数を呼ぶだけ**で使えます．

## なぜ自分のアカウントでログインして使うのか

Bluesky には「ログインなしで使える公開ホスト」もありますが，2026年9月時点で次の制約があります．

| | ログインなし（公開ホスト） | ログインあり（自分のアカウント）← **この授業の標準** |
|---|---|---|
| 事前準備 | 不要 | 無料アカウント＋**アプリパスワード**を `bsky_config.ini` に記入 |
| アクセス先 | `api.bsky.app` | 自分のサーバ `bsky.social`（そこから公開データに中継される） |
| 1回の検索 | 最大100件 | 最大100件（同じ） |
| 続きの取得（cursor） | 拒否される（403） | 使える → `search_posts_paged` で数千件まで連続取得 |
| アクセス制限 | 送信元ネットワーク単位．**学内から拒否されることがある** | アカウント単位．教室で同時に使っても影響を受けにくい |
| 取得できる範囲 | 公開投稿・公開プロフィール | 同じ（＋自分のタイムラインなど自分に紐づく情報） |
| 責任 | ― | 自分のアカウントでのアクセスとして記録される．利用規約を守る |

**変わらないこと**：取得できるのは公開投稿だけです．ログインしても，投稿者を特定・評価しない，データを再配布しない，というルールは同じです．

## 守ってほしいこと（データ倫理）

- 公開投稿であっても，投稿した人には人格があります．**特定の個人を晒す・評価する目的で使わない**．
- 集めたデータ（`data/` フォルダ）は授業内だけで使い，**外部に再配布しない**（GitHubにも上げない）．
- 発表資料には，ハンドル名や投稿URLをそのまま載せない（`anonymize()` を使う）．集計結果や，個人が特定されない形の引用にとどめる．
- APIには短時間に大量アクセスしない（関数の中で少し待ち時間を入れています）．
- アプリパスワードは自分だけの秘密です．Notebook やコードに書かない，他人に見せない．

## 0. 準備：アカウント設定ファイルを作る（初回のみ）

1. Bluesky のアカウントを持っていなければ https://bsky.app で作る（無料，メールアドレスが必要）．
2. Bluesky の **設定 → プライバシーとセキュリティ → アプリパスワード → アプリパスワードを追加** で発行し，表示された `xxxx-xxxx-xxxx-xxxx` を控える（この画面でしか表示されません）．
   通常のログインパスワードは **絶対に使わない**．
3. このフォルダの `bsky_config.example.ini` をコピーして **`bsky_config.ini`** という名前で保存し，`handle` と `app_password` を自分のものに書き換える．

```ini
[bluesky]
handle = your-name.bsky.social
app_password = xxxx-xxxx-xxxx-xxxx
```

`bsky_config.ini` は `.gitignore` に登録済みで GitHub には上がりません．
以後，`bsky_utils` を使う最初の API 呼び出しで自動的にログインします．

`bsky_utils.py` が同じフォルダにあることを確認して，読み込みます．

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from bsky_utils import *   # Bluesky用の道具箱（同じフォルダの bsky_utils.py）

pd.set_option("display.max_colwidth", 60)   # 表の本文が長すぎるときに省略する幅
pd.set_option("display.width", 160)
set_japanese_font()        # グラフの日本語が □ にならないようにする

login_status()             # 「ログイン中: @自分のハンドル」と表示されれば準備完了

> `ログインなし（公開ホスト）` と表示されたら，`bsky_config.ini` の場所（このNotebookと同じフォルダ）と内容を確認し，Kernel → Restart してからやり直してください．

## 1. キーワードで検索してみる

`search_posts("検索語")` で，その語を含む日本語の投稿を新しい順に最大100件取得します．

In [ ]:
posts = search_posts("防災", limit=20)

print("取得件数:", len(posts))
posts[["created_at_jst", "text", "like_count", "repost_count", "image_count"]]

### 表（DataFrame）の列の意味

| 列名 | 意味 |
|---|---|
| `post_uri` | 投稿を一意に示すID（`at://` で始まる）．返信の取得などに使う |
| `post_id`, `author_id` | 投稿・投稿者を匿名化した短いID（元には戻せない） |
| `author_handle` | 投稿者のハンドル名（発表資料には載せない） |
| `created_at`, `created_at_jst` | 投稿日時（UTC文字列 / 日本時間） |
| `date`, `hour`, `weekday` | 日本時間の日付・時刻（時）・曜日 |
| `text` | 本文 |
| `like_count`, `repost_count`, `reply_count`, `quote_count` | 取得時点の反応数 |
| `image_count`, `image_urls`, `image_alts` | 添付画像の枚数・URL・代替テキスト |
| `external_url`, `external_title` | 本文に付いたリンクカードのURLとタイトル |
| `query` | どの検索語で集めたか |

1件だけ取り出して全部の列を見てみましょう．

In [ ]:
posts.iloc[0]

## 2. 検索条件を変えてみる

- `sort="top"`：反応の多い順
- ハッシュタグは `"#防災"` のように指定
- `lang="en"`, `japanese_only=False` で英語の投稿

In [ ]:
# 反応の多い順
top = search_posts("防災", limit=10, sort="top")
top[["created_at_jst", "text", "like_count", "repost_count"]]

In [ ]:
# ハッシュタグで検索
tag = search_posts("#防災", limit=10)
tag[["created_at_jst", "text", "like_count"]]

In [ ]:
# 英語の投稿（言語を変える）
en = search_posts("disaster prevention", limit=5, lang="en", japanese_only=False)
en[["created_at_jst", "text", "like_count"]]

## 3. たくさん集める：続き（cursor）をたどって取得する

APIは **1回の検索で最大100件** しか返しません．ログインしていれば，結果に付いてくる「続きの目印（cursor）」を使って次の100件，その次の100件…と連続して取得できます．

`search_posts_paged(検索語, max_posts=最大件数)`

- 新しい順に取得します．`since="2026-09-01T00:00:00Z"`，`until=...` を付ければ期間内だけを取得できます．
- 1件ずつ間を空けてアクセスするので，500件で30秒ほどかかります．

In [ ]:
df_bosai = search_posts_paged("防災", max_posts=500)
print("取得件数:", len(df_bosai), " 期間:", df_bosai["created_at_jst"].min(), "〜", df_bosai["created_at_jst"].max())

比較のために，もう1つ別の話題も同じ条件で集めておきます（第3回・第4回で「話題の違い」を分析するのに使います）．

In [ ]:
df_kanko = search_posts_paged("観光", max_posts=500)

### 期間を区切って集める方法（ログインなしでも使える）

「先週1週間分」「イベントの前後3日ずつ」のように **期間で揃えて** 集めたいときは，期間を小さな窓に分割して検索する `search_posts_by_period` が便利です．1窓あたり最大100件なので，窓の中で100件に達したら `hours_per_window` を小さくします．

In [ ]:
# 例：直近2日分を12時間ごとに区切って取得（ここでは結果を確認するだけ）
df_period = search_posts_by_period("防災", days=2, hours_per_window=12)

In [ ]:
df = pd.concat([df_bosai, df_kanko], ignore_index=True)
print("合計:", len(df), "件")
df["query"].value_counts()

## 4. 集めたデータを眺める

「何件取れたか」だけでなく，**いつ・どんな投稿が・どれくらい反応されているか**を確認します．

In [ ]:
# 数値列の基本統計量
df[["like_count", "repost_count", "reply_count", "image_count"]].describe().round(1)

In [ ]:
# 時間帯別の投稿数（日本時間）
counts = df.groupby(["hour", "query"]).size().unstack(fill_value=0)
counts.plot(kind="bar", figsize=(10, 4), width=0.8)
plt.xlabel("時刻（時）")
plt.ylabel("投稿数")
plt.title("時間帯別の投稿数")
plt.tight_layout()
plt.show()

In [ ]:
# 日別の投稿数
daily = df.groupby(["date", "query"]).size().unstack(fill_value=0)
daily.plot(kind="bar", figsize=(8, 4))
plt.ylabel("投稿数")
plt.title("日別の投稿数")
plt.tight_layout()
plt.show()

In [ ]:
# 画像付き投稿の割合
rate = (df["image_count"] > 0).groupby(df["query"]).mean().round(3)
print("画像付き投稿の割合:")
print(rate)

In [ ]:
# いいねが多い投稿 上位10件
df.sort_values("like_count", ascending=False)[["query", "created_at_jst", "text", "like_count", "repost_count"]].head(10)

## 5. 特定のアカウントの投稿を集める

自治体・報道機関・企業などの **公式アカウントの投稿**を集めると，「誰が・何を・いつ発信しているか」を分析できます．

まず `search_actors("キーワード")` でアカウントを探し，ハンドル名（例: `xxxx.bsky.social`）を `get_author_posts()` に渡します．

In [ ]:
actors = search_actors("新聞", limit=10)
actors

In [ ]:
handle = actors.iloc[0]["handle"]          # ← 分析したいアカウントのハンドル名に書き換えてよい
print("プロフィール:", get_profile(handle))

author_posts = get_author_posts(handle, max_posts=100)
print("取得件数:", len(author_posts))
author_posts[["created_at_jst", "text", "like_count", "image_count"]].head(10)

## 6. 投稿への返信（スレッド）を集める

反応の多い投稿に「どんな返信が付いているか」を見ると，人々の受け止め方が分かります．

In [ ]:
# いいねが多い投稿の中から，返信が付いているものを選ぶ
candidates = top[top["reply_count"] > 0]
if len(candidates):
    target = candidates.iloc[0]
    print("元の投稿:", target["text"][:80])
    replies = get_replies(target["post_uri"])
    print("返信数:", len(replies))
    display(replies[["depth", "created_at_jst", "text", "like_count"]].head(10))
else:
    print("返信の付いた投稿がありませんでした．別の検索語で試してください．")

## 7. 保存する・読み込む

分析は何度もやり直すので，**取得したデータはCSVに保存**して，次回からは読み込んで使います．
保存すると隣に `posts.meta.json` ができ，「いつ・どの検索語で・何件」取得したかが記録されます（レポートや発表で必要な情報です）．

In [ ]:
save_posts(df, "data/posts.csv", note="第2回の演習：防災と観光を新しい順に各500件")

In [ ]:
df_loaded = load_posts("data/posts.csv")
df_loaded.head(3)

## 8. 投稿画像をダウンロードする

Bluesky の投稿には最大4枚の画像を添付できます．画像は SNS 分析の重要な材料です（「食べ物の写真が多い」「人物が写った投稿は反応が多い」「災害時にどんな写真が共有されるか」など）．第4回では，この画像に **何が写っているか** を深層学習（YOLO）で調べます．

### 画像について取得できるもの

| 項目 | 列名 | 説明 |
|---|---|---|
| 枚数 | `image_count` | 投稿に付いた画像の数（0〜4） |
| 画像URL（縮小版） | `image_thumbs` | 長辺 1000px 程度・数十KB．分析にはこれで十分 |
| 画像URL（原寸） | `image_urls` | 最大 2000px・数百KB．細部を見たいとき |
| 代替テキスト | `image_alts` | 投稿者が付けた画像の説明文（付いていないことも多い） |

複数枚のときは `|` 区切りで入っています．画像はサーバから webp 形式で届くので，扱いやすい JPEG に変換して保存します．

### 守ってほしいこと

- 画像には投稿者の著作権があります．**授業内の分析にだけ使い，再配布しない**（`data/` は Git 管理外）．
- 発表資料には画像そのものを並べず，**集計結果**（検出した物体の種類と数，明るさの分布など）で示す．必要な場合も枚数を絞り，人物が写るものは避ける．
- 一度に大量に取得しない（関数の中で1枚ごとに待ち時間を入れています）．

### 8-1. 画像付き投稿がどれくらいあるか

まず，3節で集めたデータ（`df`）で画像付き投稿の割合と枚数の分布を確認します．話題によって大きく違います．

In [ ]:
img_posts = df[df["image_count"] > 0]
print("画像付き投稿:", len(img_posts), "件 /", len(df), "件 （", round(len(img_posts) / len(df) * 100, 1), "%）")
print("画像の総数:", int(df["image_count"].sum()), "枚")

summary = df.groupby("query")["image_count"].agg(投稿数="size", 画像付き=lambda s: int((s > 0).sum()), 画像枚数="sum")
summary["画像付き割合(%)"] = (summary["画像付き"] / summary["投稿数"] * 100).round(1)
summary

In [ ]:
# 1投稿あたりの枚数の分布
df["image_count"].value_counts().sort_index().rename_axis("枚数").rename("投稿数")

### 8-2. 画像が付きやすい話題で集める

写真が多い話題（「桜」「ランチ」「旅行」「猫」など）で集めると，画像分析の練習がしやすくなります．

In [ ]:
df_photo = search_posts_paged("写真", max_posts=200)
print("画像付き投稿:", int((df_photo["image_count"] > 0).sum()), "件 / 画像の総数:", int(df_photo["image_count"].sum()), "枚")
df_photo[df_photo["image_count"] > 0][["created_at_jst", "text", "image_count", "like_count"]].head()

### 8-3. ダウンロードする

`download_images(表, out_dir=保存先, max_posts=投稿数の上限, size="thumb"|"fullsize", all_images=True|False)`

- 授業では `max_posts=20` 程度に絞ります．プロジェクトで増やすときは `scripts/download_images.py` を使います．
- 結果は **1枚1行** の表で，`out_dir/images.csv` にも保存されます（第4回の物体検出で使います）．
- 保存済みの画像は再取得しないので，同じセルを何度実行しても構いません．

In [ ]:
images = download_images(df_photo, out_dir="data/images", max_posts=20, size="thumb")
images.head(8)

### 8-4. 一覧表示する

`show_images(表, n=枚数, cols=列数, caption=列名)` で格子状に表示します．`caption="alt"` にすると代替テキストが，`caption="post_id"` にすると投稿IDが添えられます．

In [ ]:
show_images(images, n=12, cols=4, caption="alt")

### 8-5. 代替テキスト（alt）を見る

代替テキストは投稿者が画像に付けた説明文です（視覚障害のある利用者のための機能）．付いている投稿は多くありませんが，**画像の内容を文字で表したデータ**として，テキスト分析（第3回）と組み合わせられます．

In [ ]:
with_alt = images[images["alt"].fillna("") != ""]
print("代替テキスト付き:", len(with_alt), "枚 /", len(images), "枚")
with_alt[["post_id", "image_index", "alt"]].head(10)

### 8-6. 縮小版（thumb）と原寸（fullsize）の違い

1投稿だけ原寸でも取得して，大きさと容量を比べます．分析の多く（物体検出，明るさ・色の集計）は縮小版で十分です．原寸は容量が10倍近くになるので，必要なときだけ使います．

In [ ]:
one = df_photo[df_photo["image_count"] > 0].head(1)
full = download_images(one, out_dir="data/images_fullsize", max_posts=1, size="fullsize", verbose=False)
thumb = images[images["post_id"] == one.iloc[0]["post_id"]]

pd.concat([thumb.assign(種類="thumb"), full.assign(種類="fullsize")])[["種類", "image_index", "width", "height", "bytes"]]

### 8-7. 保存先の構造と，まとめてダウンロードするスクリプト

```
data/images/
├── images.csv                 ← 1枚1行の一覧（post_id, image_index, image_path, alt, width, height, bytes, size）
├── 3f2a9c1d7e5b_1.jpg         ← {post_id}_{何枚目}.jpg
├── 3f2a9c1d7e5b_2.jpg
└── ...
```

プロジェクトで数百枚を集めるときは，Anaconda Prompt からスクリプトで実行します（保存済みは飛ばすので途中で止めても再開できます）．

```bash
python scripts/download_images.py --csv data/posts.csv --out data/images --max-posts 100
python scripts/download_images.py --csv data/posts.csv --query 観光 --size fullsize --max-posts 50
python scripts/download_images.py --csv data/posts.csv --first-only --max-images 200
```

`images.csv` の `post_id` で投稿の表（本文・反応数）と結合すれば，「画像付き投稿の反応」「画像の内容と本文」のような分析ができます（第4回）．

In [ ]:
# 画像の表と投稿の表を post_id で結合する例
merged = images.merge(df_photo[["post_id", "text", "like_count", "repost_count"]], on="post_id", how="left")
merged[["post_id", "image_index", "alt", "like_count", "text"]].head()

## 9. 発展：いま流れている投稿を観測する（Jetstream）

検索は「過去の投稿から条件に合うものを探す」方法でした．
Blueskyには全投稿がリアルタイムに流れてくる **Jetstream** という仕組みもあり，
「30秒間に日本語で何件投稿されたか」「特定の語を含む投稿がどのくらいの頻度で現れるか」を観測できます．

In [ ]:
live = jetstream_collect(seconds=30, lang="ja")
live[["created_at_jst", "text"]].head(10)

In [ ]:
# 参考：Bluesky が算出しているトレンドの話題（英語圏中心）
get_trending_topics(10)

## 10. 補足：ログインの仕組みと，設定ファイルを使わない方法

- `bsky_utils` は最初の API 呼び出し時に `bsky_config.ini` を読んで自動ログインします．ログインの有効期限（約2時間）が切れても自動で更新します．
- 共用PCなどで設定ファイルを置きたくない場合は，次のように **実行時に入力** することもできます（`getpass` は入力文字を表示せず，Notebook にも残りません）．
- ログインを解除して公開ホストに戻すには `logout()` を実行します（学内ネットワークからは拒否されることがあります）．

In [ ]:
# [SKIP-TEST]
# 設定ファイルを使わずにログインする例（必要なときだけ実行）
import getpass
# login("your-name.bsky.social", getpass.getpass("アプリパスワード: "))
login_status()

## まとめ：プロジェクトで使うときのヒント

| 知りたいこと | 使う道具 |
|---|---|
| ある話題について人々が何を言っているか | `search_posts_paged("話題", max_posts=...)` |
| 期間を揃えて集めたい（先週1週間，イベント前後） | `search_posts_paged(..., since=..., until=...)` または `search_posts_by_period` |
| 2つの話題・2つの時期を比べたい | 同じ条件で2回集めて `pd.concat` |
| 自治体・企業など特定の発信者の投稿 | `search_actors` → `get_author_posts` |
| ある投稿に対する反応・意見 | `get_replies(post_uri)` |
| 投稿の量やスピードを観測したい | `jetstream_collect(seconds=...)` |
| 画像に何が写っているか | `download_images` → 第4回のYOLO |

**第2回の成果物**：`data/posts.csv` と `data/posts.meta.json`（取得条件の記録）

次回（第3回）は，このデータの本文を単語に分解して，頻出語・ワードクラウド・共起ネットワーク・時系列で可視化します．